# Ocena ludzka — Real vs Fake

Poniżej wylosowane zostanie **10 zdjęć** ze zbioru testowego (mieszanina real i fake).  
Zadanie: każdy z grupy samodzielnie ocenia każde zdjęcie jako **REAL** lub **FAKE**.  
Wyniki zapisujemy i porównujemy z predykcjami modeli w kolejnym kroku.

> Zdjęcia są wyświetlone **bez etykiet** — nie podglądaj kodu przed oceną!

In [ ]:
import os
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Ścieżka do zbioru testowego ──────────────────────────────────────────────
TEST_DIR = Path("whole/test")

FAKE_DIR = TEST_DIR / "fake"
REAL_DIR = TEST_DIR / "real"

# ── Losowanie: 5 fake + 5 real ────────────────────────────────────────────────
SEED = 42          # zmień seed aby wylosować inny zestaw
N_EACH = 5         # ile zdjęć z każdej klasy

random.seed(SEED)

fake_paths = random.sample(sorted(FAKE_DIR.glob("*.*")), N_EACH)
real_paths = random.sample(sorted(REAL_DIR.glob("*.*")), N_EACH)

# ── Tasowanie kolejności (żeby nie wiedzieć, które są z której grupy) ─────────
all_paths = fake_paths + real_paths
random.shuffle(all_paths)

# ── Zapisz prawdziwe etykiety (NIE PATRZ przed oceną!) ───────────────────────
true_labels = {p: ("fake" if p in fake_paths else "real") for p in all_paths}

print(f"Wylosowano {len(all_paths)} zdjęć (seed={SEED})")
print("Kolejność wyświetlania (indeksy 1–10) gotowa.")

In [ ]:
# ── Wyświetlenie zdjęć BEZ etykiet ───────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.patch.set_facecolor("#1a1a1a")

for idx, (ax, path) in enumerate(zip(axes.flat, all_paths), start=1):
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"#{idx}", fontsize=18, fontweight="bold",
                 color="white", pad=8)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444444")
        spine.set_linewidth(2)

plt.suptitle("Oceń każde zdjęcie: REAL czy FAKE?",
             fontsize=22, fontweight="bold", color="white", y=1.01)
plt.tight_layout(pad=1.5)
plt.show()

## Wpisz oceny — 5 osób

Każda osoba wpisuje `"real"` lub `"fake"` dla zdjęć `#1–#10`.  
Uruchom komórkę dopiero gdy wszyscy skończą.

In [ ]:
# ── Wpisz odpowiedzi każdej osoby ────────────────────────────────────────────
# Zmień wartości na "real" lub "fake"

answers = {
    "Osoba 1": {1:"?", 2:"?", 3:"?", 4:"?", 5:"?", 6:"?", 7:"?", 8:"?", 9:"?", 10:"?"},
    "Osoba 2": {1:"?", 2:"?", 3:"?", 4:"?", 5:"?", 6:"?", 7:"?", 8:"?", 9:"?", 10:"?"},
    "Osoba 3": {1:"?", 2:"?", 3:"?", 4:"?", 5:"?", 6:"?", 7:"?", 8:"?", 9:"?", 10:"?"},
    "Osoba 4": {1:"?", 2:"?", 3:"?", 4:"?", 5:"?", 6:"?", 7:"?", 8:"?", 9:"?", 10:"?"},
    "Osoba 5": {1:"?", 2:"?", 3:"?", 4:"?", 5:"?", 6:"?", 7:"?", 8:"?", 9:"?", 10:"?"},
}

for name, ans in answers.items():
    print(f"{name}: {list(ans.values())}")

## Sprawdzenie wyników — wszystkie 5 osób

Uruchom **dopiero po wpisaniu wszystkich odpowiedzi**.

In [ ]:
import numpy as np

gt = [true_labels[p] for p in all_paths]   # ground truth, lista 10 etykiet
names = list(answers.keys())
n = len(all_paths)

BORDER_OK  = "#2ecc71"
BORDER_ERR = "#e74c3c"
BORDER_UNK = "#888888"

# ── Siatka zdjęć z obramowaniami (consensus) ─────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
fig.patch.set_facecolor("#1a1a1a")

scores = {}
for name, ans in answers.items():
    correct = sum(1 for i, lbl in enumerate(gt, 1)
                  if ans.get(i, "?").lower() == lbl)
    answered = sum(1 for v in ans.values() if v != "?")
    scores[name] = (correct, answered)

for idx, (ax, path) in enumerate(zip(axes.flat, all_paths), start=1):
    img = Image.open(path).convert("RGB")
    label = gt[idx - 1]

    guesses = [answers[name].get(idx, "?").lower() for name in names]
    n_correct = sum(g == label for g in guesses if g != "?")
    n_answered = sum(g != "?" for g in guesses)

    # obramowanie zależy od tego ile osób trafiło
    if n_answered == 0:
        border = BORDER_UNK
    elif n_correct == n_answered:
        border = BORDER_OK
    elif n_correct == 0:
        border = BORDER_ERR
    else:
        border = "#f39c12"   # pomarańczowy — część trafiła

    ax.imshow(img)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border)
        spine.set_linewidth(5)

    ax.set_title(f"#{idx}  GT: {label.upper()}\n{n_correct}/{n_answered} osób trafiło",
                 fontsize=11, fontweight="bold", color=border, pad=6)
    ax.axis("off")

plt.suptitle("Wyniki grupy (zielony=wszyscy OK, pomarańczowy=część OK, czerwony=nikt)",
             fontsize=15, fontweight="bold", color="white", y=1.01)
plt.tight_layout(pad=1.5)
plt.show()

# ── Tabela wyników ────────────────────────────────────────────────────────────
print(f"\n{'Osoba':<12} {'Wynik':>8}  Odpowiedzi (1–10)")
print("─" * 60)
for name in names:
    correct, answered = scores[name]
    pct = 100 * correct / answered if answered else 0
    row = [answers[name].get(i, "?")[0].upper() for i in range(1, n + 1)]
    print(f"{name:<12} {correct}/{answered} ({pct:3.0f}%)  {' '.join(row)}")

print("─" * 60)
print(f"{'GT':<12} {'':>8}  {' '.join(l[0].upper() for l in gt)}")

# ── Wykres słupkowy ───────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
fig2.patch.set_facecolor("#1a1a1a")
ax2.set_facecolor("#1a1a1a")

pcts = [100 * scores[n][0] / scores[n][1] if scores[n][1] else 0 for n in names]
bar_colors = [BORDER_OK if p >= 70 else (BORDER_ERR if p < 50 else "#f39c12") for p in pcts]
bars = ax2.bar(names, pcts, color=bar_colors, edgecolor="#444", linewidth=1.2)
ax2.axhline(50, color="white", linestyle="--", linewidth=1, alpha=0.4, label="losowy wybór (50%)")
ax2.set_ylim(0, 105)
ax2.set_ylabel("Skuteczność [%]", color="white")
ax2.set_title("Skuteczność oceny ludzkiej", color="white", fontweight="bold")
ax2.tick_params(colors="white")
for spine in ax2.spines.values():
    spine.set_edgecolor("#444")
for bar, pct in zip(bars, pcts):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
             f"{pct:.0f}%", ha="center", va="bottom", color="white", fontweight="bold")
ax2.legend(facecolor="#333", labelcolor="white")
plt.tight_layout()
plt.show()

## Porównanie: GT vs Człowiek (konsensus) vs Modele

Dla każdego zdjęcia zestawia:
- **GT** – prawdziwa etykieta,
- **Człowiek** – demokratyczny wybór większości (np. 3/5 → real),
- **ResNet-18** – predykcja modelu PyTorch (resnet18_best.pth),
- **ResNet-50** – predykcja modelu Keras (detektor_deepfake_resnet.keras).


In [ ]:
import torch
import torchvision.transforms as T
import torchvision.models as tvm
import torch.nn as nn
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

# ── Urządzenie PyTorch ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── ResNet-18 (PyTorch) ───────────────────────────────────────────────────────
rn18 = tvm.resnet18(weights=None)
rn18.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(rn18.fc.in_features, 2))
rn18.load_state_dict(torch.load("resnet18_best.pth", map_location=device))
rn18 = rn18.to(device).eval()

rn18_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_rn18(path):
    """Zwraca ('real'|'fake', prawdopodobieństwo) dla pliku path."""
    img = Image.open(path).convert("RGB")
    x = rn18_transforms(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = rn18(x)
        probs = torch.softmax(logits, dim=1)[0]   # [p_fake, p_real]
    # ImageFolder sortuje klasy alfabetycznie: 0=fake, 1=real
    pred_idx = probs.argmax().item()
    return ("fake" if pred_idx == 0 else "real"), probs[pred_idx].item()

# ── ResNet-50 (Keras / TensorFlow) ───────────────────────────────────────────
rn50 = tf.keras.models.load_model("detektor_deepfake_resnet.keras")

def predict_rn50(path):
    """Zwraca ('real'|'fake', prawdopodobieństwo) dla pliku path."""
    img = tf.keras.utils.load_img(path, target_size=(224, 224))
    x = tf.keras.utils.img_to_array(img)
    x = tf.expand_dims(x, 0)                       # batch dim
    prob_real = float(rn50.predict(x, verbose=0)[0][0])
    # sigmoid: >0.5 → klasa 1 (real), <=0.5 → klasa 0 (fake)
    if prob_real > 0.5:
        return "real", prob_real
    else:
        return "fake", 1.0 - prob_real

# ── Konsensus ludzki (demokratyczny wybór większości) ────────────────────────
def human_consensus(img_idx):
    """Dla zdjęcia nr img_idx (1-based) zwraca etykietę z większości głosów."""
    votes = [answers[name].get(img_idx, "?").lower() for name in answers]
    votes = [v for v in votes if v in ("real", "fake")]
    if not votes:
        return "?"
    return "real" if votes.count("real") >= votes.count("fake") else "fake"

# ── Kolory etykiet ────────────────────────────────────────────────────────────
LABEL_COLOR = {"real": "#2ecc71", "fake": "#e74c3c", "?": "#888888"}

def colored_label(label, prediction):
    """Kolor zielony gdy predykcja == GT, czerwony gdy różna."""
    return LABEL_COLOR.get(prediction, "#888888")

# ── Siatka porównawcza ────────────────────────────────────────────────────────
n_imgs = len(all_paths)
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
fig.patch.set_facecolor("#1a1a1a")

for idx, (ax, path) in enumerate(zip(axes.flat, all_paths), start=1):
    gt   = true_labels[path]
    hum  = human_consensus(idx)
    r18_pred, r18_conf = predict_rn18(path)
    r50_pred, r50_conf = predict_rn50(path)

    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")

    # ── Obramowanie: zielone gdy WSZYSCY 3 (człowiek, rn18, rn50) trafiają ──
    all_correct = (hum == gt) and (r18_pred == gt) and (r50_pred == gt)
    none_correct = (hum != gt) and (r18_pred != gt) and (r50_pred != gt)
    border_color = "#2ecc71" if all_correct else ("#e74c3c" if none_correct else "#f39c12")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(5)

    def mark(pred, gt):
        return "✓" if pred == gt else "✗"

    title = (
        f"#{idx}  GT: {gt.upper()}\n"
        f"Człowiek: {hum.upper()} {mark(hum, gt)}\n"
        f"ResNet-18: {r18_pred.upper()} {mark(r18_pred, gt)}  ({r18_conf:.0%})\n"
        f"ResNet-50: {r50_pred.upper()} {mark(r50_pred, gt)}  ({r50_conf:.0%})"
    )
    ax.set_title(title, fontsize=9.5, fontweight="bold", color="white",
                 pad=6, linespacing=1.4,
                 fontfamily="monospace")

plt.suptitle(
    "Porównanie: GT vs Człowiek (większość) vs ResNet-18 vs ResNet-50\n"
    "Obramowanie: zielone = wszyscy trafili  |  pomarańczowe = część  |  czerwone = nikt",
    fontsize=13, fontweight="bold", color="white", y=1.02
)
plt.tight_layout(pad=1.5)
plt.show()

# ── Tabela podsumowująca ──────────────────────────────────────────────────────
print(f"\n{'#':<4} {'GT':<6} {'Człowiek':<10} {'ResNet-18':<12} {'ResNet-50':<12}")
print("─" * 46)
human_ok = rn18_ok = rn50_ok = 0
for idx, path in enumerate(all_paths, start=1):
    gt = true_labels[path]
    hum = human_consensus(idx)
    r18_pred, _ = predict_rn18(path)
    r50_pred, _ = predict_rn50(path)
    human_ok += hum == gt
    rn18_ok  += r18_pred == gt
    rn50_ok  += r50_pred == gt
    print(f"{idx:<4} {gt.upper():<6} "
          f"{hum.upper()+' '+('✓' if hum==gt else '✗'):<10} "
          f"{r18_pred.upper()+' '+('✓' if r18_pred==gt else '✗'):<12} "
          f"{r50_pred.upper()+' '+('✓' if r50_pred==gt else '✗'):<12}")
print("─" * 46)
print(f"{'Acc':<4} {'':6} {human_ok}/{n_imgs} ({100*human_ok/n_imgs:.0f}%)   "
      f"{rn18_ok}/{n_imgs} ({100*rn18_ok/n_imgs:.0f}%)   "
      f"{rn50_ok}/{n_imgs} ({100*rn50_ok/n_imgs:.0f}%)")
